# NLP II — Transformer / BERT・Hugging Face

Transformer と BERT の考え方、Hugging Face でのファインチューニング、LLM 活用の概観を学びます。

> **Kaggle Tutorial** 対応ノートブック（`/learn/deep-learning/07-transformers`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# transformers の確認・自動インストールとサンプル英文リストを用意します
import subprocess
import sys

try:
    import transformers  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers"], check=True)

import torch

texts = [
    "Pretrained transformers capture context effectively for text competitions.",
    "Fine-tuning with a small learning rate stabilizes downstream performance.",
]

## Transformer と BERT

**Transformer** は自己注意（self-attention）で文中の語の関係を捉えるアーキテクチャです。
**BERT** は大量テキストで事前学習された Transformer で、これを各タスクに
**転移学習（ファインチューニング）**して使うのが NLP の主流です。

## Hugging Face でファインチューニング

`transformers` ライブラリで、事前学習済みモデルとトークナイザを簡単に呼べます。

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2)

enc = tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
# あとは通常の PyTorch 学習ループ、または Trainer で学習

In [ ]:
model.eval()
with torch.no_grad():
    probs = torch.softmax(model(**enc).logits, dim=-1)
print("input_ids shape:", enc["input_ids"].shape)
print("predicted probabilities:\n", probs.cpu().numpy().round(4))

ファインチューニングでは小さめの学習率（例 2e-5）と短いエポック、
検証での早期終了が定番です。GPU メモリに応じてバッチサイズや `max_length` を調整します。

## LLM 活用の概観

近年は大規模言語モデル（LLM）を、ゼロショット/few-shot 推論や埋め込み生成、
データ拡張（擬似データ生成）に使うアプローチも増えています。ただしコンペでは
**規約（外部データ・モデルの利用可否）**を必ず確認します。

> **Warning:** 事前学習モデルやデータには**ライセンス**と**コンペ規約**があります。利用可否・商用条件・ 外部データ規約を確認してから使いましょう（L6 の引用・倫理も参照）。

> **Tip:** BERT 系は強力ですが学習コストが高めです。**TF-IDF + 線形（前レッスン）とアンサンブル**すると、 誤りの相関が低く伸びやすい組み合わせになります。

## 次のステップ

次は時系列予測の考え方と、検証でのリーク回避を学びます。